# Measure per-frame intensity percentiles

For every cells/hyb round declared in `round_info.csv` -- plus any ad hoc,
undeclared test-reimage folder sitting alongside a declared one (e.g.
`H01_old`, `H01_old_1` next to a declared `H01`; see
`MERci.common.metadata.discover_ad_hoc_round_dirs`) -- and every FOV,
computes an exact per-frame intensity table:

    frame, z, color, min, p25, p50, p75, p95, max

Runs as a SLURM array job, one task per individual FOV movie file (one
round/hyb x one FOV), tolerant of missing/partial data: a round not yet
imaged, a FOV not yet transferred, or a file still mid-write simply stays
pending and is picked up on a later re-run of this notebook rather than
crashing the whole pass (see section 5). Results are cached as parquet
under `analysis/cache/12_measure_intensity_percentiles/per_fov/` --
re-running this notebook only computes what's missing.


## 1 — Setup

In [ ]:
import os
import sys
import csv
import json
from pathlib import Path

import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/after_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata, discover_ad_hoc_round_dirs
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.progress_display       import ProgressReporter
from MERci.acquisition.configs    import find_frame_table_for_hal_config
from MERci.analysis.fov           import measure_intensity_percentiles, load_all_intensity_percentiles
from MERci.acquisition.cluster_submit import (
    build_intensity_percentiles_array_script, submit_sbatch, is_job_active,
)

print(f"SAMPLE_DIR : {SAMPLE_DIR}")


## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX = ".zarr"

# Which percentiles to compute (in addition to min/max)
PERCENTILES = (25, 50, 75, 95)

# Submit the per-FOV-movie jobs as a SLURM array (recommended -- a serial
# scan over a real experiment takes far too long) vs. run locally/serially,
# only practical for a small dataset or a quick local test.
USE_SLURM_ARRAY         = True
SLURM_ARRAY_CONCURRENCY = 50
SLURM_MEM                = "1gb"
SLURM_TIME               = "00:10:00"
# ^ from a real benchmark against LT066_sample_01's lineage-tracing data
#   (215-frame, 2304x2304 uint16 FOVs): ~45-55s wall time, ~250-260MB peak
#   RSS per file -- see MERci.acquisition.cluster_submit.
#   build_intensity_percentiles_array_script's own docstring for the note.

print(f"Sample name    : {SAMPLE_NAME}")
print(f"Positions tag  : {POSITIONS_TAG}")
print(f"Percentiles    : {PERCENTILES}")


## 3 — Resolve config, metadata, and every round-like folder (declared + ad hoc)

`round_series` maps a human-readable folder label (e.g. `"cells"`, `"H01"`,
or an ad hoc `"H01_old"`) to the `SeriesInfo` used to resolve that folder's
per-FOV file paths -- declared rounds come straight from `round_info.csv`
via `ExperimentMetadata`; ad hoc folders are discovered by scanning for
undeclared `<round>_old[_N]` siblings next to a declared round's own
folder, borrowing that round's series pattern/hal_config (same imaging
recipe, just a re-image).

In [ ]:
config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
)

meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)

NOTEBOOK_NAME = "12_measure_intensity_percentiles"
cache_dir  = config.analysis_dir / "cache" / NOTEBOOK_NAME
output_dir = cache_dir / "per_fov"
cache_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

round_series = {}
for rid in meta.valid_round_ids():
    for s in meta.series_for_round(rid):
        label = s.candidate_dirs[0].name if s.candidate_dirs else f"round{rid}"
        round_series[label] = s
declared_labels = set(round_series)

ad_hoc_series = discover_ad_hoc_round_dirs(meta)
round_series.update(ad_hoc_series)

print(f"Declared rounds : {len(declared_labels)} -- {sorted(declared_labels)}")
print(f"Ad hoc folders  : {len(ad_hoc_series)}" + (f" -- {sorted(ad_hoc_series)}" if ad_hoc_series else ""))
print(f"FOVs            : {meta.n_fovs}")
print(f"Cache           : {cache_dir}")


## 4 — Resolve each round-like folder's frame table

Every round/hyb needs its own `z`/`color` mapping (the frame table tied to
its `hal_config`); an ad hoc folder borrows its base round's frame table
(same `hal_config`, since it's the same imaging recipe re-run).

In [ ]:
frame_table_paths = {}
for label, s in round_series.items():
    if not s.hal_config:
        print(f"  [skip] {label}: no hal_config on its series row -- can't resolve a frame table.")
        continue
    ft_path = find_frame_table_for_hal_config(config.settings_dir / s.hal_config, config.metadata_dir)
    if ft_path is None:
        print(f"  [skip] {label}: no frame table found for hal_config={s.hal_config!r}.")
        continue
    frame_table_paths[label] = ft_path

print(f"{len(frame_table_paths)} / {len(round_series)} round(s)/folder(s) have a resolvable frame table.")


## 5 — Compute (SLURM array, one task per FOV movie file)

Each array task (`cli_measure_intensity_percentiles.py`) computes one
image file's full intensity-percentile table and writes it as parquet to
`output_dir` -- a (round/hyb, FOV) pair is skipped on a re-run once its
cache file already exists. A round/FOV whose raw file isn't on disk yet
(round not imaged, FOV not transferred) is simply left out of the pending
list -- not an error -- and will be picked up automatically once the file
appears.

In [ ]:
def fov_cache_path(round_label, fov_id):
    return output_dir / f"{round_label}__fov{fov_id:04d}_intensity_percentiles.parquet"

to_compute = []   # (image_path, frame_table_path, round_label, fov_id, output_path)
n_missing = 0
for label, ft_path in frame_table_paths.items():
    series = round_series[label]
    for fov_id in sorted(meta.fovs):
        fpath = series.resolve_path(fov_id, config.image_suffix)
        if not fpath.exists():
            n_missing += 1
            continue
        out_path = fov_cache_path(label, fov_id)
        if out_path.exists():
            continue
        to_compute.append((str(fpath), str(ft_path), label, fov_id, str(out_path)))

print(f"{n_missing} expected file(s) not yet on disk (rounds/FOVs still being imaged or "
      f"transferred -- normal mid-acquisition, not an error).")
print(f"{len(to_compute)} file(s) pending computation (not yet cached).")


In [ ]:
if to_compute and USE_SLURM_ARRAY:
    job_sentinel = cache_dir / "intensity_percentiles_job.json"
    cached_job = json.loads(job_sentinel.read_text()) if job_sentinel.exists() else None

    if (cached_job is not None and cached_job.get("n_pending") == len(to_compute)
            and is_job_active(cached_job["job_id"])):
        print(f"SLURM array job {cached_job['job_id']} is still active "
              f"({len(to_compute)} file(s) pending) -- re-run this cell later once it finishes.")
    else:
        manifest_path = cache_dir / "intensity_percentiles_manifest.csv"
        with open(manifest_path, "w", newline="") as fh:
            writer = csv.writer(fh)
            for row in to_compute:
                writer.writerow(row)

        script_path = cache_dir / "intensity_percentiles.sh"
        build_intensity_percentiles_array_script(
            sample_dir=SAMPLE_DIR, manifest_path=manifest_path,
            n_pending=len(to_compute), output_path=script_path, percentiles=PERCENTILES,
            array_concurrency=SLURM_ARRAY_CONCURRENCY, mem=SLURM_MEM, time=SLURM_TIME,
        )
        job_id = submit_sbatch(script_path)
        if job_id is not None:
            job_sentinel.write_text(json.dumps({"job_id": job_id, "n_pending": len(to_compute)}))
            print(f"Submitted SLURM array job {job_id} for {len(to_compute)} file(s) -- "
                  f"re-run this cell later once it finishes to load the results.")
        else:
            print("sbatch submission failed (see the logged error above) -- fix the issue and re-run this cell.")
elif to_compute:
    reporter = ProgressReporter(total=len(to_compute), label="Measuring intensity percentiles (local)")
    for image_path, ft_path, label, fov_id, out_path in reporter.wrap(to_compute):
        frame_table = pd.read_csv(ft_path, index_col=0)
        df = measure_intensity_percentiles(Path(image_path), frame_table, Path(out_path), percentiles=PERCENTILES)
        df.insert(0, "fov_id", fov_id)
        df.insert(0, "round_label", label)
        df.to_parquet(out_path, index=False)
else:
    print("All discovered round/FOV movie files already cached.")


## 6 — Results

In [ ]:
results = load_all_intensity_percentiles(output_dir)
if results.empty:
    print("No results cached yet.")
else:
    n_files = results.groupby(["round_label", "fov_id"]).ngroups
    print(f"Loaded {len(results)} row(s) across {n_files} round x FOV file(s), "
          f"{results['round_label'].nunique()} round(s), {results['fov_id'].nunique()} FOV(s).")
results
